<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 7 · Fine-Tuning and LoRA

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/07-finetuning-lora.ipynb)

**From Traditional NLP to Modern LLMs** · Day 2 · 120 minutes (55 briefing, 55 lab, 10 debrief)

Turning a pretrained language model into an instruction follower, efficiently.

**By the end of this notebook you can:**

- Turn a pretrained causal LM into an instruction follower
- Apply LoRA and explain why it works
- Choose decoding settings and evaluate generated text

**Stack:** Hugging Face, PEFT, PyTorch

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "07-finetuning-lora"
_CONTENT_SHA = "181d81ee8b921c5c"
_EXERCISES = {1: ('LoRALinear',), 2: ('merged_weight',), 3: ('format_chat',), 4: ('build_labels',), 5: ('lora_param_count', 'lora_config'), 6: ('rouge_n',)}
_HINTS = {1, 4, 6}  # exercises with a folded Hint cell above their TODO cell
_PATHS = {'offline': 'offline stand-ins and test doubles: checks that the code runs, not what a model does', 'open': 'real open models from the Hugging Face Hub, no API keys', 'keyed': 'commercial APIs with keys'}
_FALLBACK_NOTE = 'On a CPU runtime, FAST runs a shorter fine-tune of the same SmolLM2-135M. There is no path without the Hugging Face Hub'
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_OFFLINE_FLAGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
            if n in _HINTS:
                print(f"[workshop] Stuck? Open the folded Hint above TODO {n} first.")
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Read the assertion above,"
                f" fix TODO {n}, or go on with workshop.use_reference({n!r})."
            )
            if n in _HINTS:
                print(f"[workshop] The folded Hint above TODO {n} says what to check.")

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        path = self.path()
        # _PATHS comes from readiness.paths in _variables.yml; the offline entry says that
        # such numbers check the code, not a model.
        print(f"  What ran: {_PATHS[path]}.")
        if path == "open" and _FALLBACK_NOTE:
            print(f"  Read the numbers with this in mind: {_FALLBACK_NOTE}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def path(self):
        """offline, keyed or open: the same rule as scripts/add_run_record.py's path_of."""
        ns = _get_ipython().user_ns
        if self.fallbacks or any(_os.environ.get(flag) for flag in _OFFLINE_FLAGS):
            return "offline"
        if ns.get("PROVIDER") in ("openai", "anthropic") or ns.get("JEV_PATH") == "keyed":
            return "keyed"
        return "open"

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

**Runtime.** Use a GPU: *Runtime → Change runtime type → T4 GPU*. On a CPU runtime the lab switches itself to a smaller `FAST` run (fewer examples and steps) so that it still finishes. No API key is needed.

**What you build.** A LoRA layer by hand, then a LoRA fine-tune of a small pretrained language model (SmolLM2-135M, the *base* model, not the instruction-tuned one) on instruction data, measured before and after.

**How the lab works.** There are six exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one short function or class;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your code is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

The setup cell installs pinned versions of `transformers`, `peft` and `accelerate` if the runtime has different ones. It does not reinstall PyTorch: Colab's CUDA build is kept.

In [ ]:
# Setup: pinned installs, imports, seeds, device and run size.
import importlib.metadata
import sys

# Versions this lab was written against (current releases on 2026-10-04). torch is not
# pinned: reinstalling it on Colab takes minutes and the lab uses only stable torch APIs.
PINS = {"transformers": "5.18.0", "peft": "0.21.2", "accelerate": "1.15.0"}


def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


_needed = [f"{name}=={version}" for name, version in PINS.items() if installed_version(name) != version]
if _needed:
    %pip install -q {" ".join(_needed)}

import json
import math
import os
import random
import re
import shutil
import textwrap
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import peft
import torch
import torch.nn as nn
import transformers
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
transformers.logging.set_verbosity_error()  # hide routine warnings from generate()
transformers.utils.logging.disable_progress_bar()  # no download and weight-loading bars

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IN_COLAB = "google.colab" in sys.modules
FAST = DEVICE == "cpu"  # the smaller run; set to False to force the full run on a CPU

# Run sizes. The full run is the T4 path; FAST is the CPU path that CI executes.
RUN = (
    dict(n_train=300, n_heldout=50, n_generate=10, steps=40, stretch_steps=15)
    if FAST
    else dict(n_train=2000, n_heldout=200, n_generate=50, steps=250, stretch_steps=100)
)
BATCH_SIZE = 16  # training examples per step
MAX_NEW_TOKENS = 64  # length limit for every generated reply
LR = 5e-4  # adapter learning rate (AdamW)

# TEST-ONLY SWITCH. Leave it unset. The repository's tests set the environment variable
# NLP_LLMS_LAB07_OFFLINE on machines that cannot reach huggingface.co; the lab then runs on a
# RANDOMLY INITIALIZED stand-in model and synthetic data (see the hidden cell under "Data and
# model"). Every number printed in that mode is marked and is not a result.
OFFLINE = os.environ.get("NLP_LLMS_LAB07_OFFLINE", "")  # "", "tiny" or "smollm2-shape"
assert OFFLINE in ("", "tiny", "smollm2-shape"), OFFLINE
NOTE = "   [TEST-ONLY random stand-in model: not a result]" if OFFLINE else ""

NAVY, ACCENT = "#16324f", "#b3541e"  # plot colors
LAB_START = time.time()
TIMES = {}  # stage -> seconds, printed at the end of the core path
print(f"torch {torch.__version__} · transformers {transformers.__version__} · peft {peft.__version__}")
print(f"device {DEVICE} · FAST {FAST} · run sizes {RUN}")
if OFFLINE:
    print(f"\n*** OFFLINE TEST MODE ({OFFLINE}): random stand-in model and synthetic data. ***")

## Data and Model

| | |
|---|---|
| Model | `HuggingFaceTB/SmolLM2-135M`, the base model, pinned to one revision. Apache-2.0. 134.5 million parameters, 30 transformer blocks, hidden size 576 |
| Data | Databricks Dolly 15k: 15,011 instructions with human-written responses, in 8 categories. CC BY-SA 3.0 |
| Subset | prompt at most 128 tokens and response at most 64 tokens after formatting; shuffled with seed 0; 200 held-out and 2,000 training examples (FAST: the first 50 and 300 of these) |
| Display prompts | the first 5 held-out examples, shown before and after fine-tuning |

**Attribution.** Databricks Dolly 15k, © Databricks, CC BY-SA 3.0, with material from Wikipedia. The subset this notebook builds is an adaptation and stays under CC BY-SA 3.0. The fine-tuned adapter is not distributed by the workshop.

The first cell is the workshop's standard loader (`data/README.md` in the repository), with one function added for Dolly. It checks the file against a SHA-256 hash, so everyone gets the same bytes and therefore the same subset.

In [ ]:
import csv, gzip, hashlib, io, os, urllib.request
from pathlib import Path


def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


def load_dolly():
    """Databricks Dolly 15k (CC BY-SA 3.0). Returns a list of 15,011 dicts with the fields
    instruction, context, response and category, in file order."""
    blob = fetch(
        "databricks-dolly-15k.jsonl",
        [
            "https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a/databricks-dolly-15k.jsonl",
        ],
        "2df9083338b4abd6bceb5635764dab5d833b393b55759dffb0959b6fcbf794ec",
    )
    return [json.loads(line) for line in blob.decode("utf-8").splitlines() if line.strip()]

In [ ]:
#@title Test-only: offline stand-in for the model and the data (ignore this cell) { display-mode: "form" }
# Used ONLY when NLP_LLMS_LAB07_OFFLINE is set (see the setup cell), by the repository's tests on
# a machine that cannot reach huggingface.co. It builds:
#   - a tokenizer trained on synthetic text, with SmolLM2's three special tokens at the same ids
#     (<|endoftext|> 0, <|im_start|> 1, <|im_end|> 2);
#   - a RANDOMLY INITIALIZED Llama model: "tiny" (2 blocks, hidden size 64) or "smollm2-shape"
#     (exactly SmolLM2-135M's shapes, to check parameter counts and CPU run time);
#   - 3,000 synthetic instruction records with Dolly's fields.
# The rest of the notebook runs unchanged on them. Nothing measured on this stand-in is a result.


def synthetic_records(n=3000, seed=1234):
    rng = random.Random(seed)
    capitals = {"France": "Paris", "Japan": "Tokyo", "Kenya": "Nairobi", "Peru": "Lima",
                "Canada": "Ottawa", "Egypt": "Cairo", "Norway": "Oslo", "India": "New Delhi"}
    names = ["Ana", "Ben", "Chen", "Dara", "Eli", "Fatima", "Goran", "Hana"]
    jobs = ["teacher", "nurse", "pilot", "farmer", "chef", "lawyer"]
    records = []
    for _ in range(n):
        kind = rng.randrange(3)
        if kind == 0:
            country = rng.choice(sorted(capitals))
            rec = dict(instruction=f"What is the capital of {country}?", context="",
                       response=f"The capital of {country} is {capitals[country]}.", category="open_qa")
        elif kind == 1:
            a, b = rng.randrange(20), rng.randrange(20)
            rec = dict(instruction=f"What is {a} plus {b}?", context="",
                       response=f"{a} plus {b} is {a + b}.", category="open_qa")
        else:
            name, job, city = rng.choice(names), rng.choice(jobs), rng.choice(sorted(capitals.values()))
            rec = dict(instruction="Who is this text about, and what do they do?",
                       context=f"{name} moved to {city} last year and now works as a {job}.",
                       response=f"It is about {name}, who works as a {job} in {city}.", category="closed_qa")
        records.append(rec)
    return records


def build_offline_stand_in(kind):
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
    from transformers import LlamaConfig, LlamaForCausalLM, PreTrainedTokenizerFast

    records = synthetic_records()
    bpe = Tokenizer(models.BPE())
    bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    bpe.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=600, special_tokens=["<|endoftext|>", "<|im_start|>", "<|im_end|>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    bpe.train_from_iterator(
        [r["instruction"] + "\n\n" + r["context"] + "\n" + r["response"] for r in records] + ["user", "assistant"],
        trainer,
    )
    tok = PreTrainedTokenizerFast(
        tokenizer_object=bpe, bos_token="<|endoftext|>", eos_token="<|endoftext|>",
        unk_token="<|endoftext|>", additional_special_tokens=["<|im_start|>", "<|im_end|>"],
    )
    if kind == "tiny":
        shape = dict(vocab_size=len(tok), hidden_size=64, intermediate_size=128,
                     num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2)
    else:  # SmolLM2-135M's shapes (134,515,008 parameters), random weights
        shape = dict(vocab_size=49152, hidden_size=576, intermediate_size=1536,
                     num_hidden_layers=30, num_attention_heads=9, num_key_value_heads=3)
    config = LlamaConfig(**shape, max_position_embeddings=2048, tie_word_embeddings=True,
                         bos_token_id=0, eos_token_id=0)
    torch.manual_seed(SEED)
    out = Path(f"lab07-offline-{kind}")
    LlamaForCausalLM(config).save_pretrained(out)
    tok.save_pretrained(out)
    return str(out), records

In [ ]:
# Must match models.instruct_base and models.instruct_base_revision in _variables.yml.
MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
MODEL_REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"

# The ChatML chat template of the briefing (section 2). The base tokenizer ships without one.
CHATML_TEMPLATE = (
    "{%- for message in messages %}"
    "{{- '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}{{- '<|im_start|>assistant\n' }}{%- endif %}"
)

started = time.time()
print("Loading the model and the data; the first run downloads them, which can take a minute ...")
if OFFLINE:
    MODEL_SOURCE, dolly = build_offline_stand_in(OFFLINE)  # test-only: see the hidden cell above
    LOAD_KWARGS = {}
else:
    MODEL_SOURCE, LOAD_KWARGS = MODEL_ID, {"revision": MODEL_REVISION}
    dolly = load_dolly()


def load_base():
    """A fresh copy of the pretrained base model, in float32, on DEVICE."""
    # float32, not the default bfloat16: in bfloat16 the merge check is off and a CPU step is slow.
    base = AutoModelForCausalLM.from_pretrained(MODEL_SOURCE, dtype=torch.float32, **LOAD_KWARGS)
    return base.to(DEVICE).eval()


tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, **LOAD_KWARGS)
print("Chat template shipped with the base tokenizer:", tokenizer.chat_template)
tokenizer.chat_template = CHATML_TEMPLATE
tokenizer.pad_token = tokenizer.eos_token  # no pad token in the base tokenizer; padding is never scored
tokenizer.padding_side = "right"  # training pads on the right; generation switches to the left
IM_START = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")
for special in ("<|im_start|>", "<|im_end|>"):  # each must be ONE existing token: no resizing
    assert len(tokenizer(special, add_special_tokens=False).input_ids) == 1, special

model = load_base()
n_params = sum(p.numel() for p in model.parameters())
TIMES["load model and data"] = time.time() - started
print(f"{len(dolly):,} records · <|im_start|> = {IM_START}, <|im_end|> = {IM_END}, pad = {tokenizer.pad_token_id}")
print(f"{type(model).__name__}: {n_params:,} parameters, {model.config.num_hidden_layers} blocks, "
      f"hidden size {model.config.hidden_size} ({TIMES['load model and data']:.1f} s){NOTE}")

In [ ]:
PROMPT_MAX, RESPONSE_MAX = 128, 64  # tokens, after formatting; the response count includes <|im_end|>


def user_messages(ex):
    return [{"role": "user", "content": ex["user"]}]


def full_messages(ex):
    return user_messages(ex) + [{"role": "assistant", "content": ex["response"]}]


def make_subset(records, tokenizer, n_heldout=200, n_train=2000, seed=SEED):
    """The fixed Dolly subset: filter by length, shuffle with `seed`, split. Returns (train, held_out, stats)."""
    examples = []
    for i, rec in enumerate(records):
        instruction, context, response = (rec[k].strip() for k in ("instruction", "context", "response"))
        if instruction and response:
            user = instruction + ("\n\n" + context if context else "")  # context after a blank line
            examples.append({"dolly_index": i, "category": rec["category"], "user": user, "response": response})
    prompts = tokenizer.apply_chat_template(
        [user_messages(ex) for ex in examples], tokenize=False, add_generation_prompt=True
    )
    prompt_len = [len(ids) for ids in tokenizer(prompts, add_special_tokens=False).input_ids]
    response_len = [
        len(ids) + 1 for ids in tokenizer([ex["response"] for ex in examples], add_special_tokens=False).input_ids
    ]  # + 1 for <|im_end|>
    kept = [ex for ex, p, q in zip(examples, prompt_len, response_len) if p <= PROMPT_MAX and q <= RESPONSE_MAX]
    random.Random(seed).shuffle(kept)
    assert len(kept) >= n_heldout + n_train, f"only {len(kept)} records pass the length filter"
    stats = {"records": len(records), "non_empty": len(examples), "pass_filter": len(kept)}
    return kept[n_heldout : n_heldout + n_train], kept[:n_heldout], stats


train_full, heldout_full, subset_stats = make_subset(dolly, tokenizer)
train_set = train_full[: RUN["n_train"]]
held_out = heldout_full[: RUN["n_heldout"]]
gen_set = held_out[: RUN["n_generate"]]  # replies are generated for these
display_set = held_out[:5]  # shown side by side before and after
print(subset_stats, f"· using {len(train_set)} train, {len(held_out)} held-out, {len(gen_set)} generated")
print("Categories in the training set:", dict(Counter(ex["category"] for ex in train_set).most_common()))

if not OFFLINE:  # write the subset so it can be committed under data/ (see data/README.md)
    with open("lab07_dolly_subset.jsonl", "w", encoding="utf-8") as f:
        for split, rows in (("heldout", heldout_full), ("train", train_full)):
            for ex in rows:
                f.write(json.dumps({"split": split, **ex}, ensure_ascii=False) + "\n")
    blob = Path("lab07_dolly_subset.jsonl").read_bytes()
    print(f"Wrote lab07_dolly_subset.jsonl: {len(blob):,} bytes, sha256 {hashlib.sha256(blob).hexdigest()}")

# Step 0 · A Language Model Continues; It Does Not Answer (3 minutes)

`generate_reply` formats a conversation with the chat template, appends the **generation prompt** `<|im_start|>assistant\n`, and decodes greedily until the model emits `<|im_end|>` or reaches `MAX_NEW_TOKENS`. It also stops at `<|endoftext|>`, the token that ended every document in pretraining, because a model may end its reply with that habit instead of with `<|im_end|>`. It is the helper from section 7 of the briefing. Lab 8 reuses it.

**Predict.** The base model was trained only to continue ordinary text. Given `<|im_start|>user\n…<|im_end|>\n<|im_start|>assistant\n`, will it answer and stop?

In [ ]:
def generate_reply(model, tokenizer, messages, **gen_kwargs) -> str:
    """Generate the assistant's reply to `messages` (a list of {"role", "content"} dicts)."""
    # Stop at <|im_end|>, the template's end of a reply, or at <|endoftext|>, the end-of-document
    # token the base model learned in pretraining.
    stop_ids = [tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    settings = dict(max_new_tokens=64, do_sample=False, eos_token_id=stop_ids, pad_token_id=tokenizer.pad_token_id)
    settings.update(gen_kwargs)
    with torch.no_grad():
        output = model.generate(**inputs, **settings)
    new = output[0, inputs["input_ids"].shape[1] :].tolist()  # generate returns prompt + continuation
    ends = [i for i, t in enumerate(new) if t in stop_ids]
    return tokenizer.decode(new[: ends[0]] if ends else new, skip_special_tokens=True)


def show(text, width=100, indent="    "):
    print(textwrap.indent(textwrap.fill(text.replace("\n", " ⏎ "), width), indent))


started = time.time()
for ex in display_set:
    print("USER:")
    show(ex["user"][:300])
    print("BASE MODEL:")
    show(generate_reply(model, tokenizer, user_messages(ex), max_new_tokens=MAX_NEW_TOKENS) or "(empty)")
    print()
print(f"({time.time() - started:.1f} s){NOTE}")

**Explain.** Compare with your prediction. Look for replies that repeat the request, drift into unrelated text, or run to the length limit instead of stopping. The base model has seen `<|im_start|>` and `<|im_end|>` rarely or never in context, so the template means little to it yet. Fine-tuning has to teach two things: what a reply looks like, and when to stop.

# Part A · LoRA by Hand

## Exercise 1 · The LoRA Layer (10 minutes)

LoRA keeps a pretrained linear layer $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$ frozen and adds a trainable low-rank update (briefing, eqs. `lora` and `lora-forward`):

$$
W = W_0 + \frac{\alpha}{r}\, B A, \qquad h = W_0 x + \frac{\alpha}{r}\, B (A x),
$$

with $A \in \mathbb{R}^{r \times d_{\text{in}}}$ (small random values), $B \in \mathbb{R}^{d_{\text{out}} \times r}$ (zeros), rank $r$ and a fixed scale $\alpha / r$. In code, $W_0$ is `base.weight` and the scale is `scale`. Batch size is written `batch_size`, never `B`.

**Predict.** For `nn.Linear(576, 192)`, the shape of a value projection in this lab's model, and $r = 8$: how many trainable parameters does the adapter have? How many does the frozen layer have, counting its bias?

**Task.** Complete `__init__` and `forward`.

- Freeze every parameter of `base` (`p.requires_grad_(False)`).
- `A`: an `nn.Parameter` of shape `(r, d_in)`, `torch.randn(...) * 0.01`. `B`: an `nn.Parameter` of shape `(d_out, r)`, zeros.
- `forward(x)` for `x` of shape `(..., d_in)`: the base output plus `scale` times `x` through `A` then `B`.

<details>
<summary>Hint for TODO 1</summary>

`B` starts at zero so that the adapter adds exactly nothing at initialization; `A` is the random one (briefing eq. `lora`). Freeze every parameter of `base`, its bias included, or the trainable count comes out $d_{\text{out}}$ too high. In `forward`, take `x` through `A` first, down to $r$ numbers, then through `B`, back up to $d_{\text{out}}$, and multiply by `self.scale` (eq. `lora-forward`).

</details>

In [ ]:
# TODO 1: complete __init__ and forward.
class LoRALinear(nn.Module):
    """A frozen nn.Linear plus a trainable rank-r update: h = W_0 x + (alpha / r) B A x."""

    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base
        ...  # freeze W_0 and the bias
        d_out, d_in = base.weight.shape
        self.A = ...  # (r, d_in), small random
        self.B = ...  # (d_out, r), zero
        self.scale = ...  # alpha / r

    def forward(self, x):  # x: (..., d_in)
        raise NotImplementedError("TODO 1")

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
class LoRALinear(nn.Module):
    """A frozen nn.Linear plus a trainable rank-r update: h = W_0 x + (alpha / r) B A x."""

    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)  # freeze W_0 and the bias
        d_out, d_in = base.weight.shape
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)  # (r, d_in), small random
        self.B = nn.Parameter(torch.zeros(d_out, r))  # (d_out, r), zero
        self.scale = alpha / r

    def forward(self, x):  # x: (..., d_in)
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)  # (..., d_out)

<details>
<summary>Why this solution works</summary>

`x @ self.A.T` maps each input vector to $r$ numbers, shape `(..., r)`, and `@ self.B.T` maps them back up to `(..., d_out)`. That is $B(Ax)$ for every vector in the batch at once: eq. `lora-forward`. The full $d_{\text{out}} \times d_{\text{in}}$ matrix $BA$ is never formed, which is cheaper when $r$ is small.

Only `A` and `B` are `nn.Parameter`s with `requires_grad=True`, so an optimizer built from the trainable parameters touches nothing else. The count is $r \cdot d_{\text{in}} + d_{\text{out}} \cdot r = r(d_{\text{in}} + d_{\text{out}})$ (eq. `lora-count`): $8 \times (576 + 192) = 6{,}144$, against $576 \times 192 + 192 = 110{,}784$ frozen. With `B` all zeros the bypass adds exactly zero, so the output is bit-for-bit the base layer's.

</details>

In [ ]:
# Checkpoint 1
workshop.checkpoint(1)
torch.manual_seed(SEED)
_base = nn.Linear(576, 192)  # a value projection's shape; it has a bias here to check the bias is frozen too
_layer = LoRALinear(_base, r=8, alpha=16)
_x = torch.randn(4, 10, 576)  # (batch_size, T, d_in)
assert _layer.A.shape == (8, 576) and _layer.B.shape == (192, 8), (
    f"A must be (r, d_in) = (8, 576) and B (d_out, r) = (192, 8); got {tuple(_layer.A.shape)} and {tuple(_layer.B.shape)}")
assert _layer.A.abs().sum() > 0 and not _layer.B.any(), (
    f"A starts random and B at zero; got sum |A| = {_layer.A.abs().sum().item():.3g} "
    f"and {int((_layer.B != 0).sum())} nonzero entries in B")
assert math.isclose(_layer.scale, 16 / 8), "scale = alpha / r"
assert _layer(_x).shape == (4, 10, 192), (
    f"the output must have shape (batch_size, T, d_out) = (4, 10, 192), got {tuple(_layer(_x).shape)}")
assert torch.equal(_layer(_x), _base(_x)), "at initialization (B = 0) the output must equal the base layer's"
_trainable = sum(p.numel() for p in _layer.parameters() if p.requires_grad)
assert _trainable == 8 * (576 + 192), (
    f"trainable parameters: {_trainable}, expected r * (d_in + d_out) = 6144: freeze every parameter of base, "
    "the bias included")
assert not _base.weight.requires_grad and not _base.bias.requires_grad, "W_0 and the bias must be frozen"
print(f"Checkpoint 1 passed: {_trainable:,} trainable, "
      f"{sum(p.numel() for p in _base.parameters()):,} frozen")

**Run: the gradients at initialization.** The cell below takes one gradient step's worth of gradients on a random regression target. By the chain rule (eq. `lora-grad`), with $g = \partial \mathcal{L} / \partial h$:

$$
\frac{\partial \mathcal{L}}{\partial B} = \frac{\alpha}{r}\, g\, (A x)^\top, \qquad \frac{\partial \mathcal{L}}{\partial A} = \frac{\alpha}{r}\, (B^\top g)\, x^\top .
$$

**Predict.** Which of the two gradients is exactly zero at initialization? Does that mean training cannot start?

In [ ]:
_target = torch.randn(4, 10, 192)
_loss = ((_layer(_x) - _target) ** 2).mean()
_loss.backward()  # autograd computes eq. lora-grad
print(f"norm of dL/dA: {_layer.A.grad.norm():.6f}")
print(f"norm of dL/dB: {_layer.B.grad.norm():.6f}")
print(f"gradient of W_0: {_base.weight.grad}  (frozen, so none is computed)")
assert _layer.A.grad.abs().max() == 0 and _layer.B.grad.abs().max() > 0

**Explain.** The gradient of $A$ contains $B^\top g$, and $B = 0$, so it is zero. The gradient of $B$ contains $Ax$, which is not zero because $A$ is random. The first optimizer step moves $B$; from then on $A$ gets gradient too. If both started at zero, both gradients would stay zero forever.

## Exercise 2 · Merging the Update (5 minutes)

After training, the adapter is just a matrix of the same shape as $W_0$ (eq. `lora-merge`):

$$
W_{\text{merged}} = W_0 + \frac{\alpha}{r}\, B A .
$$

The next cell trains a fresh `LoRALinear` (rank 4) on a toy regression whose true weight is $W_0$ plus a rank-2 change, so that $B$ is no longer zero.

**Predict.** A plain `nn.Linear` holding $W_{\text{merged}}$ and the original bias: will its output match the adapter's exactly, approximately, or not at all?

**Task.** Write `merged_weight(layer)`, which returns $W_{\text{merged}}$ with shape `(d_out, d_in)`. It is attached to the class as a method.

In [ ]:
# Provided: train a LoRA layer on a toy regression so that B != 0.
torch.manual_seed(SEED)
toy_base = nn.Linear(64, 32)  # W_0: (32, 64)
delta = 0.1 * torch.randn(32, 2) @ torch.randn(2, 64)  # the change to learn, rank 2
X = torch.randn(1024, 64)  # (n, d_in)
Y = (X @ (toy_base.weight + delta).T + toy_base.bias).detach()  # targets from W_0 + delta
toy = LoRALinear(toy_base, r=4, alpha=8)
opt = torch.optim.Adam([toy.A, toy.B], lr=1e-2)
for step in range(300):
    loss = ((toy(X) - Y) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step in (0, 299):
        print(f"step {step:>3}  mean squared error {loss.item():.6f}")
assert toy.B.abs().max() > 0

In [ ]:
# TODO 2: complete this function.
def merged_weight(layer):
    """W_merged = W_0 + (alpha / r) B A for a LoRALinear `layer`. Shape (d_out, d_in)."""
    raise NotImplementedError("TODO 2")

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def merged_weight(layer):
    """W_merged = W_0 + (alpha / r) B A for a LoRALinear `layer`. Shape (d_out, d_in)."""
    return layer.base.weight + layer.scale * (layer.B @ layer.A)  # (d_out, r) @ (r, d_in)

<details>
<summary>Why this solution works</summary>

$(d_{\text{out}} \times r)$ times $(r \times d_{\text{in}})$ gives a $d_{\text{out}} \times d_{\text{in}}$ matrix, the shape of $W_0$. Since matrix multiplication distributes over addition, $(W_0 + sBA)x = W_0 x + s\,B(Ax)$ for every $x$: the merged layer computes the same function as the adapter. The two differ only by floating-point rounding, because the sums happen in a different order. After merging there are no extra parameters and no extra computation at inference.

</details>

In [ ]:
# Checkpoint 2
workshop.checkpoint(2)
LoRALinear.merged_weight = lambda layer: merged_weight(layer)  # calls whichever merged_weight is bound: yours, or the reference after use_reference(2)
with torch.no_grad():
    W = toy.merged_weight()
    assert W.shape == toy_base.weight.shape, (
        f"W_merged must have the shape of W_0, {tuple(toy_base.weight.shape)}; got {tuple(W.shape)}: "
        "B @ A is (d_out, r) @ (r, d_in)")
    plain = nn.Linear(64, 32)
    plain.weight.copy_(W)
    plain.bias.copy_(toy_base.bias)
    max_diff = (plain(X) - toy(X)).abs().max().item()
    recovered = ((W - toy_base.weight - delta).norm() / delta.norm()).item()
print(f"max |merged layer - adapter| = {max_diff:.2e}")
print(f"relative error of the learned update against the true rank-2 change: {recovered:.4f}")
assert torch.allclose(plain(X), toy(X), atol=1e-5), (
    f"the merged layer must reproduce the adapter; max |difference| {max_diff:.2e}: "
    "did you include the scale alpha / r?")
print("Checkpoint 2 passed")

# Part B · Instruction Data

## Exercise 3 · The Chat Template (6 minutes)

A **chat template** turns a list of messages, each with a `role` and a `content`, into the one string the model sees. ChatML marks each message with two special tokens: `<|im_start|>` opens it and `<|im_end|>` closes it. With `add_generation_prompt=True` the string ends with the opening of an assistant message, `<|im_start|>assistant\n`, so that the likely continuation is a reply.

The tokenizer already holds this template as a Jinja string (set in the loading cell above). You write the same thing in Python.

**Predict.** How many newline characters are in the formatted string for one user message with `add_generation_prompt=True` (the content itself has none)?

**Task.** Complete `format_chat(messages, add_generation_prompt)`. Each message becomes `<|im_start|>` + role + `\n` + content + `<|im_end|>` + `\n`.

In [ ]:
# TODO 3: complete this function.
def format_chat(messages, add_generation_prompt=False):
    """The ChatML string for a list of {"role", "content"} messages."""
    text = ""
    ...
    return text

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def format_chat(messages, add_generation_prompt=False):
    """The ChatML string for a list of {"role", "content"} messages."""
    text = ""
    for m in messages:
        text += "<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>\n"
    if add_generation_prompt:
        text += "<|im_start|>assistant\n"  # the generation prompt
    return text

<details>
<summary>Why this solution works</summary>

A chat template is string formatting and nothing more (briefing, section 2). The roles exist for the model only as the tokens between `<|im_start|>` and the newline. Because the template is part of the model's input format, a model tuned with it expects exactly this layout, newlines included, which is why the checkpoint asks for exact equality and why the template is saved with the tokenizer at the end of the lab.

The answer to the prediction is three: one after the role, one after `<|im_end|>`, one after `assistant`.

</details>

In [ ]:
# Checkpoint 3
workshop.checkpoint(3)
_conv = [
    {"role": "system", "content": "You answer in one sentence."},
    {"role": "user", "content": "Name the capital of France."},
    {"role": "assistant", "content": "Paris."},
]
for _messages in (_conv, _conv[1:2]):
    for _flag in (False, True):
        _ours = format_chat(_messages, add_generation_prompt=_flag)
        _theirs = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=_flag)
        assert _ours == _theirs, f"add_generation_prompt={_flag}:\n{_ours!r}\n{_theirs!r}"
_ids = tokenizer(format_chat(_conv), add_special_tokens=False).input_ids
assert _ids.count(IM_START) == 3 and _ids.count(IM_END) == 3, "each special token must be one token"
print("Checkpoint 3 passed. One user message with the generation prompt:")
print(repr(format_chat(_conv[1:2], add_generation_prompt=True)))

## Exercise 4 · The Response Mask and the Labels (8 minutes)

Instruction tuning scores only the reply (briefing, eq. `masked-loss`). For a formatted example $w_1, \dots, w_T$ with **response mask** $m_t = 1$ on the response tokens and the closing `<|im_end|>`, and $0$ elsewhere:

$$
\mathcal{L}_{\text{resp}} = -\frac{1}{\sum_t m_t} \sum_{t=1}^{T} m_t \log P_\theta(w_t \mid w_{<t}).
$$

The provided `encode_example` builds the token ids and `response_mask` for one example: the prompt (user message plus generation prompt) gets 0, the response and its `<|im_end|>` get 1. A training sequence ends at that `<|im_end|>`: the newline the template writes after it is never generated, so it is left out and not scored.

The mask reaches the loss through the labels. PyTorch's cross-entropy, and therefore the model's `.loss`, skips every position whose label is `-100`. The model shifts the labels by one position itself.

**Predict.** For the example *Name the capital of France.* → *Paris.*, how many positions are scored?

**Task.** Complete `build_labels(input_ids, response_mask)`: a copy of `input_ids` with `-100` wherever `response_mask` is 0. Both have shape `(batch_size, T)`. Padding has mask 0, so it is not scored either.

In [ ]:
def encode_example(ex, tokenizer):
    """Token ids of one formatted example and its response mask: two lists of equal length."""
    prompt = tokenizer.apply_chat_template(user_messages(ex), tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    response_ids = tokenizer(ex["response"], add_special_tokens=False).input_ids + [IM_END]
    return prompt_ids + response_ids, [0] * len(prompt_ids) + [1] * len(response_ids)


_ex = {"user": "Name the capital of France.", "response": "Paris."}
_ids, _mask = encode_example(_ex, tokenizer)
# The training sequence is the chat template's string, ending at the final <|im_end|>.
assert tokenizer.decode(_ids) == format_chat(full_messages(_ex))[: -len("\n")]
for _tok, _m in zip(tokenizer.convert_ids_to_tokens(_ids), _mask):
    print(f"{_m}  {_tok!r}")

<details>
<summary>Hint for TODO 4</summary>

Clone `input_ids` before writing $-100$: the model still reads every prompt token, and only the loss skips them (eq. `masked-loss`). The $-100$ goes where `response_mask` is 0; the response and its closing `<|im_end|>` keep their IDs. Do not shift the labels yourself: the model's loss shifts them by one position.

</details>

In [ ]:
# TODO 4: complete this function.
def build_labels(input_ids, response_mask):
    """Labels for the masked loss: input_ids where response_mask is 1, -100 elsewhere. (batch_size, T)."""
    raise NotImplementedError("TODO 4")

In [ ]:
#@title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
def build_labels(input_ids, response_mask):
    """Labels for the masked loss: input_ids where response_mask is 1, -100 elsewhere. (batch_size, T)."""
    labels = input_ids.clone()  # never modify input_ids: the model still reads every token
    labels[response_mask == 0] = -100  # prompt and padding are not scored
    return labels

<details>
<summary>Why this solution works</summary>

The model still reads the whole prompt: every response token is predicted from $w_{<t}$, which includes the instruction. Only the sum in eq. `masked-loss` skips the prompt positions, and `-100` is how that is said to the loss. The model's loss shifts the labels by one, so position $t$'s label is scored against the prediction made at $t - 1$, and divides by the number of scored positions: the $\frac{1}{\sum_t m_t}$ of the equation.

`<|im_end|>` is in the sum on purpose: it is how the model learns to stop. For the example, the scored positions are the tokens of `Paris.` plus `<|im_end|>`.

</details>

In [ ]:
# Checkpoint 4
workshop.checkpoint(4)
_ids_t, _mask_t = torch.tensor([_ids]), torch.tensor([_mask])  # (1, T)
_labels = build_labels(_ids_t, _mask_t)
assert _labels.shape == _ids_t.shape and _labels.dtype == torch.long, (
    f"labels must match input_ids, shape {tuple(_ids_t.shape)} and dtype torch.int64; "
    f"got {tuple(_labels.shape)} and {_labels.dtype}")
assert torch.equal(_ids_t, torch.tensor([_ids])), "build_labels must not modify input_ids"
_n_scored = int((_labels != -100).sum())
_expected = len(tokenizer("Paris.", add_special_tokens=False).input_ids) + 1  # + <|im_end|>
assert _n_scored == _expected, (
    f"{_n_scored} scored positions, expected {_expected} (the reply's tokens plus <|im_end|>): "
    "-100 goes where response_mask is 0")
assert _labels[0, -1] == IM_END, (
    f"the closing <|im_end|> (ID {IM_END}) must be scored, but the last label is {_labels[0, -1].item()}: "
    "do not shift the labels; the model shifts them")
print("Scored text:", repr(tokenizer.decode(_labels[0][_labels[0] != -100])))

# The model's loss equals a hand computation of eq. masked-loss.
with torch.no_grad():
    _out = model(input_ids=_ids_t.to(DEVICE), labels=_labels.to(DEVICE))
    _logp = torch.log_softmax(_out.logits[0, :-1].float(), dim=-1)  # (T-1, V): predicts tokens 2..T
    _token_logp = _logp.gather(-1, _ids_t[0, 1:, None].to(DEVICE)).squeeze(-1)  # (T-1,)
    _m = _mask_t[0, 1:].to(DEVICE).float()
    _hand = -(_token_logp * _m).sum() / _m.sum()
print(f"model .loss {_out.loss.item():.6f} · hand computation {_hand.item():.6f}{NOTE}")
assert abs(_out.loss.item() - _hand.item()) < 1e-4, (
    f"model .loss {_out.loss.item():.6f} differs from the hand computation {_hand.item():.6f}: "
    "labels must equal input_ids wherever response_mask is 1")
print(f"Checkpoint 4 passed: {_n_scored} scored positions")

**Run: the "before" measurement.** Everything below uses the same held-out examples, the same template, greedy decoding and the same `MAX_NEW_TOKENS`. It must run **before** Exercise 5, because `get_peft_model` modifies the model in place.

- **Response loss** and **response perplexity** (eqs. `masked-loss` and `ppl-resp`), averaged over all response tokens of the held-out set.
- **How each reply ends**: with `<|im_end|>` (the **stop rate**: the reply ended as the template says), with `<|endoftext|>` (the base model's end-of-document habit), or at the length limit.

In [ ]:
ENCODED = {}  # cache: id(example) -> (ids, response_mask)


def collate(examples):
    """Right-padded batch: input_ids, attention_mask, response_mask, each (batch_size, T)."""
    encoded = []
    for ex in examples:
        if id(ex) not in ENCODED:
            ENCODED[id(ex)] = encode_example(ex, tokenizer)
        encoded.append(ENCODED[id(ex)])
    T = max(len(ids) for ids, _ in encoded)
    input_ids = torch.full((len(encoded), T), tokenizer.pad_token_id)
    attention_mask = torch.zeros((len(encoded), T), dtype=torch.long)
    response_mask = torch.zeros((len(encoded), T), dtype=torch.long)
    for i, (ids, mask) in enumerate(encoded):
        input_ids[i, : len(ids)] = torch.tensor(ids)
        attention_mask[i, : len(ids)] = 1
        response_mask[i, : len(ids)] = torch.tensor(mask)
    return input_ids, attention_mask, response_mask


@torch.no_grad()
def response_loss(model, examples, batch_size=8):
    """Mean negative log-likelihood over all response tokens of `examples` (eq. masked-loss)."""
    model.eval()
    total, count = 0.0, 0
    for i in range(0, len(examples), batch_size):
        input_ids, attention_mask, response_mask = (t.to(DEVICE) for t in collate(examples[i : i + batch_size]))
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.float()  # (batch_size, T, V)
        nll = nn.functional.cross_entropy(
            logits[:, :-1].transpose(1, 2), input_ids[:, 1:], reduction="none"
        )  # (batch_size, T-1): position t predicts token t+1
        mask = response_mask[:, 1:].float()
        total += (nll * mask).sum().item()
        count += int(mask.sum().item())
    return total / count, count


@torch.no_grad()
def generate_batch(model, examples, batch_size=16, **gen_kwargs):
    """Greedy replies to many examples. Returns (replies, endings): endings[i] is "im_end",
    "endoftext" or "length", the way reply i ended."""
    model.eval()
    stop_ids = {IM_END: "im_end", tokenizer.eos_token_id: "endoftext"}
    settings = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, eos_token_id=list(stop_ids),
                    pad_token_id=tokenizer.pad_token_id)
    settings.update(gen_kwargs)
    replies, endings = [], []
    tokenizer.padding_side = "left"  # every prompt must end at the same position
    try:
        for i in range(0, len(examples), batch_size):
            prompts = tokenizer.apply_chat_template(
                [user_messages(ex) for ex in examples[i : i + batch_size]], tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
            output = model.generate(**inputs, **settings)
            for row in output[:, inputs["input_ids"].shape[1] :].tolist():
                end = next((i for i, t in enumerate(row) if t in stop_ids), None)  # first stop token
                replies.append(tokenizer.decode(row if end is None else row[:end], skip_special_tokens=True))
                endings.append("length" if end is None else stop_ids[row[end]])
    finally:
        tokenizer.padding_side = "right"
    return replies, endings


def ending_rates(endings):
    return {kind: endings.count(kind) / len(endings) for kind in ("im_end", "endoftext", "length")}


started = time.time()
loss_before, n_resp_tokens = response_loss(model, held_out)
before_replies, before_endings = generate_batch(model, gen_set)
_probe = collate(held_out[:2])  # kept to check the adapter changes nothing at initialization
with torch.no_grad():
    probe_logits_before = model(input_ids=_probe[0].to(DEVICE), attention_mask=_probe[1].to(DEVICE)).logits
ends_before = ending_rates(before_endings)
TIMES["before evaluation"] = time.time() - started
print(f"Base model on {len(held_out)} held-out examples ({n_resp_tokens:,} response tokens){NOTE}")
print(f"  response loss {loss_before:.4f} nats · response perplexity {math.exp(loss_before):,.2f}")
print(f"  {len(gen_set)} replies ended with <|im_end|> {ends_before['im_end']:.0%} · <|endoftext|> "
      f"{ends_before['endoftext']:.0%} · length limit {ends_before['length']:.0%} ({TIMES['before evaluation']:.1f} s)")

# Part C · LoRA With `peft`

## Exercise 5 · Configure LoRA and Count Its Parameters (10 minutes)

`peft` wraps every targeted linear layer with the layer of Exercise 1. `LoraConfig` takes the rank `r`, the constant `lora_alpha` ($\alpha$; the scale is `lora_alpha / r`), and `target_modules`, matched against the end of each module's name. In this model the query and value projections of attention are named `q_proj` and `v_proj`. The cell below prints their shapes.

**Predict.** With $r = 8$ on `q_proj` and `v_proj` in all blocks, how many trainable parameters will `peft` report? Use eq. `lora-count`, $r(d_{\text{in}} + d_{\text{out}})$ per layer, and write the number down before running anything.

**Task.**

1. Write `lora_param_count(model, r, target_modules)`: eq. `lora-count` summed over every `nn.Linear` whose name ends with one of `target_modules` (use `module.in_features` and `module.out_features`).
2. Complete `lora_config`: rank 8, `lora_alpha = 2 * r`, targets `q_proj` and `v_proj`, `task_type="CAUSAL_LM"`.

In [ ]:
for _name, _module in model.named_modules():
    if _name.endswith(("layers.0.self_attn.q_proj", "layers.0.self_attn.v_proj")):
        print(f"{_name}: {_module}")
print(f"... and the same in each of the {model.config.num_hidden_layers} blocks")

In [ ]:
# TODO 5: complete the function and the config.
def lora_param_count(model, r, target_modules):
    """Trainable parameters LoRA adds: r * (d_in + d_out) for each targeted nn.Linear."""
    raise NotImplementedError("TODO 5")


r = 8
lora_config = LoraConfig(
    r=...,
    lora_alpha=...,
    target_modules=...,
    lora_dropout=0.0,
    task_type="CAUSAL_LM",
)

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def lora_param_count(model, r, target_modules):
    """Trainable parameters LoRA adds: r * (d_in + d_out) for each targeted nn.Linear."""
    total = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and name.split(".")[-1] in target_modules:
            total += r * (module.in_features + module.out_features)  # eq. lora-count
    return total


r = 8
lora_config = workshop.solution_value(5, "lora_config", LoraConfig(
    r=r,
    lora_alpha=2 * r,  # scale = lora_alpha / r = 2
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM",
))

<details>
<summary>Why this solution works</summary>

Each block contributes $8 \times (576 + 576) = 9{,}216$ for `q_proj` and $8 \times (576 + 192) = 6{,}144$ for `v_proj`, so $15{,}360$ per block and $30 \times 15{,}360 = 460{,}800$ in all: about 0.34% of the model. `v_proj` is smaller than `q_proj` because the model shares each key and value head among three query heads (grouped-query attention), so keys and values have 192 dimensions, not 576.

`peft` counts `requires_grad` parameters after wrapping, so the two numbers agree only if your formula is right and the config targets exactly these layers. `init_lora_weights=True`, the default, gives the initialization of Exercise 1: $B = 0$, $A$ random.

</details>

In [ ]:
# Checkpoint 5
workshop.checkpoint(5)
predicted = lora_param_count(model, lora_config.r, lora_config.target_modules)
torch.manual_seed(SEED)  # A is initialized randomly
model = get_peft_model(model, lora_config)  # wraps q_proj and v_proj in place, freezes everything else
model.print_trainable_parameters()
trainable, total = model.get_nb_trainable_parameters()
print(f"predicted {predicted:,}{NOTE}")
assert lora_config.r == 8 and lora_config.lora_alpha == 16, (
    f"rank 8 and lora_alpha = 2 * r = 16; got r = {lora_config.r}, lora_alpha = {lora_config.lora_alpha}")
assert set(lora_config.target_modules) == {"q_proj", "v_proj"}, (
    f"target exactly q_proj and v_proj; got {lora_config.target_modules}")
assert trainable == predicted, (
    f"peft reports {trainable:,}, your formula gives {predicted:,}: each targeted nn.Linear adds r * (d_in + d_out)")
if not OFFLINE or OFFLINE == "smollm2-shape":
    assert trainable == 460_800, trainable  # SmolLM2-135M, r = 8 on q_proj and v_proj
with torch.no_grad():  # B = 0, so the wrapped model is still exactly the base model
    _logits = model(input_ids=_probe[0].to(DEVICE), attention_mask=_probe[1].to(DEVICE)).logits
assert torch.allclose(_logits, probe_logits_before, atol=1e-5), "the untrained adapter changed the output"
print("Checkpoint 5 passed")

**Run: training.** The loop is provided. Each step takes `BATCH_SIZE` training examples, builds the labels with your `build_labels`, and takes an AdamW step on the adapter parameters only. The learning rate warms up over 10 steps and then decays linearly to zero. The full run takes 250 steps of 16 examples: exactly two passes over the 2,000 training examples.

In [ ]:
def train_lora(model, examples, steps, lr=LR, batch_size=BATCH_SIZE, seed=SEED, log_every=10):
    """Train the trainable parameters of `model` on `examples` with the masked loss. Returns the losses."""
    params = [p for p in model.parameters() if p.requires_grad]  # only A and B
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    warmup = 10
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warmup, (steps - s) / max(steps - warmup, 1)))
    gen = torch.Generator().manual_seed(seed)  # the same batches for every run with this seed
    order, losses = [], []
    model.train()
    for step in range(steps):
        if len(order) < batch_size:
            order += torch.randperm(len(examples), generator=gen).tolist()
        batch, order = [examples[j] for j in order[:batch_size]], order[batch_size:]
        input_ids, attention_mask, response_mask = (t.to(DEVICE) for t in collate(batch))
        labels = build_labels(input_ids, response_mask)
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        sched.step()
        opt.zero_grad(set_to_none=True)
        losses.append(loss.item())
        if log_every and (step % log_every == 0 or step == steps - 1):
            print(f"step {step:>4}  loss {loss.item():.4f}")
    model.eval()
    return losses


started = time.time()
train_losses = train_lora(model, train_set, RUN["steps"])
TIMES["LoRA training"] = time.time() - started
print(f"{RUN['steps']} steps in {TIMES['LoRA training']:.1f} s{NOTE}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(train_losses, color=NAVY, linewidth=1.5)
ax.set_xlabel("step")
ax.set_ylabel("training response loss (nats)")
ax.set_title("LoRA fine-tuning" + (" [TEST-ONLY stand-in]" if OFFLINE else ""))
plt.show()

**Run: the "after" measurement**, on the same held-out examples with the same settings.

In [ ]:
workshop.checkpoint(5, label="after training")
started = time.time()
loss_after, _ = response_loss(model, held_out)
after_replies, after_endings = generate_batch(model, gen_set)
ends_after = ending_rates(after_endings)
TIMES["after evaluation"] = time.time() - started

print(f"{len(held_out)} held-out examples, greedy decoding, max {MAX_NEW_TOKENS} new tokens{NOTE}\n")
print(f"{'':<8}{'response loss':>15}{'perplexity':>13}  | replies ending with {'<|im_end|>':>11}{'<|endoftext|>':>15}{'length':>8}")
for name, loss, ends in (("before", loss_before, ends_before), ("after", loss_after, ends_after)):
    print(f"{name:<8}{loss:>15.4f}{math.exp(loss):>13,.2f}  | {'':>19} {ends['im_end']:>11.0%}"
          f"{ends['endoftext']:>15.0%}{ends['length']:>8.0%}")

# TARGET margins, not measurements: to be set from at least three seeds on a T4 (full run) and
# on a CPU runtime (FAST run). On the offline stand-in only the direction is checked.
LOSS_MARGIN = 0.0 if OFFLINE else (0.2 if FAST else 0.5)
assert loss_after < loss_before - LOSS_MARGIN, (loss_before, loss_after, LOSS_MARGIN)
print(f"\nCheckpoint passed: held-out response loss fell by more than {LOSS_MARGIN} nats")

In [ ]:
print(f"The 5 display prompts, before and after (greedy){NOTE}\n")
for ex, before, after in zip(display_set, before_replies, after_replies):
    print("USER:")
    show(ex["user"][:300])
    print("REFERENCE:")
    show(ex["response"])
    print("BEFORE:")
    show(before or "(empty)")
    print("AFTER:")
    show(after or "(empty)")
    print("-" * 104)

**Explain.** Three numbers and five examples:

- The response loss measures how much probability the model gives the *reference* replies. A fall means the model now expects replies of this kind, starting with the template's tokens. It says nothing directly about what the model writes.
- The stop rate (replies ending with `<|im_end|>`) measures one behavior that the loss trains through a single token. Compare it with Step 0. If many replies end with `<|endoftext|>` instead, the model has learned *that* a reply ends but still ends it with its pretraining habit. The template's end token was rare in pretraining, its output weights are frozen here, and the adapter sits only in attention, so raising its probability can be slow. The stretch section compares adapting every linear layer.
- Read the five pairs. Is the "after" reply on topic? Is it correct? A 135M model can learn the format of a reply quickly and still get facts wrong.

Only 0.34% of the parameters moved. Exercise 6 adds an overlap score on the generated text.

# Part D · Decoding and Evaluating Generated Text

## Exercise 6 · ROUGE-$n$ and Decoding Settings (8 minutes)

**ROUGE-$n$** is n-gram recall against a reference (briefing, eq. `rouge`). With $c_{\text{cand}}(g)$ and $c_{\text{ref}}(g)$ the counts of n-gram $g$ in the candidate and the reference:

$$
\text{ROUGE-}n = \frac{\sum_{g} \min\big(c_{\text{cand}}(g),\, c_{\text{ref}}(g)\big)}{\sum_{g} c_{\text{ref}}(g)},
$$

both sums over the n-grams of the reference. The helper `rouge_tokens` lowercases and keeps runs of letters and digits.

**Predict.** Candidate *the the the the*, reference *the cat sat on the mat*. What is ROUGE-1? What would it be without the $\min$?

**Task.** Complete `rouge_n(candidate, reference, n)`. If the reference has fewer than $n$ tokens, return 0.0.

In [ ]:
def rouge_tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def ngrams(tokens, n):
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))

<details>
<summary>Hint for TODO 6</summary>

ROUGE-$n$ is recall: divide by the number of n-grams in the reference, not in the candidate (briefing eq. `rouge`). Clip each n-gram's credit at its count in the reference, so *the the the the* against a reference with two *the*s scores $2/6$, not $4/6$. The helpers `rouge_tokens` and `ngrams` do the tokenizing and the counting.

</details>

In [ ]:
# TODO 6: complete this function.
def rouge_n(candidate, reference, n=1):
    """ROUGE-n recall of `candidate` against `reference` (two strings)."""
    raise NotImplementedError("TODO 6")

In [ ]:
#@title Solution 6 — try it yourself first { display-mode: "form" }
@workshop.solution(6)
def rouge_n(candidate, reference, n=1):
    """ROUGE-n recall of `candidate` against `reference` (two strings)."""
    ref = ngrams(rouge_tokens(reference), n)
    if not ref:
        return 0.0
    cand = ngrams(rouge_tokens(candidate), n)
    overlap = sum(min(cand[g], count) for g, count in ref.items())  # clipped counts
    return overlap / sum(ref.values())

<details>
<summary>Why this solution works</summary>

The sum runs over the reference's n-grams, so the denominator is the number of n-grams in the reference: this is recall. The $\min$ clips each n-gram's credit at the number of times the reference contains it. Without it, *the the the the* would score 4/6 against a reference with two *the*s; with it, 2/6. ROUGE stays a surface measure: a correct paraphrase scores low, and a fluent wrong answer that reuses the reference's words scores high.

</details>

In [ ]:
# Checkpoint 6
workshop.checkpoint(6)
assert math.isclose(rouge_n("the the the the", "the cat sat on the mat", 1), 2 / 6), (
    f"ROUGE-1 of 'the the the the' against 'the cat sat on the mat' is 2/6, got "
    f"{rouge_n('the the the the', 'the cat sat on the mat', 1)}: clip each count at the reference's, "
    "and divide by the reference's n-grams")
assert rouge_n("the cat sat on the mat", "the cat sat on the mat", 2) == 1.0, (
    f"a candidate equal to the reference scores 1.0, got {rouge_n('the cat sat on the mat', 'the cat sat on the mat', 2)}")
assert math.isclose(rouge_n("the cat", "the cat sat on the mat", 2), 1 / 5), (
    f"'the cat' matches 1 of the reference's 5 bigrams, so 1/5, got {rouge_n('the cat', 'the cat sat on the mat', 2)}")
assert rouge_n("Paris", "The capital is Paris.", 1) == 0.25, (
    f"'Paris' matches 1 of the 4 reference words, so 0.25, got {rouge_n('Paris', 'The capital is Paris.', 1)}: "
    "tokenize both strings with rouge_tokens")
assert rouge_n("anything", "short", 2) == 0.0, "a reference shorter than n has no n-grams"
assert rouge_n("", "the cat", 1) == 0.0, f"an empty candidate scores 0.0, got {rouge_n('', 'the cat', 1)}"
print("Checkpoint 6 passed\n")

refs = [ex["response"] for ex in gen_set]
rouge1_before = float(np.mean([rouge_n(c, ref, 1) for c, ref in zip(before_replies, refs)]))
rouge1_after = float(np.mean([rouge_n(c, ref, 1) for c, ref in zip(after_replies, refs)]))
print(f"Mean ROUGE-1 on {len(gen_set)} held-out replies: before {rouge1_before:.3f} · after {rouge1_after:.3f}{NOTE}")

**Run: decoding settings.** Greedy decoding is deterministic, which is why every measurement above uses it. Now sample instead.

**Predict.** With `do_sample=True`, will three runs with different seeds give the same reply? Will a temperature of 1.5 give more or less sensible text than 0.7?

Edit `SAMPLING` and re-run the cell. Set `top_k` and `top_p` yourself: with `do_sample=True`, `generate` applies top-$k$ with $k = 50$ when you do not, and `temperature` is ignored when `do_sample=False`. `top_k=0` turns top-$k$ off.

In [ ]:
SAMPLING = dict(do_sample=True, temperature=0.7, top_p=0.9, top_k=0)  # edit me

prompt = user_messages(display_set[0])
print("USER:")
show(display_set[0]["user"][:300])
greedy = [generate_reply(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS) for _ in range(2)]
print("GREEDY (twice):")
for reply in greedy:
    show(reply or "(empty)")
assert greedy[0] == greedy[1], "greedy decoding must be deterministic"
print(f"SAMPLED {SAMPLING}:")
for seed in range(3):
    torch.manual_seed(seed)
    show(generate_reply(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS, **SAMPLING) or "(empty)")
print(NOTE)

**Explain.** Greedy gives the same reply every time, so a change in a score measured with it is a change in the model. Sampled replies differ between seeds; a higher temperature flattens the distribution and lets unlikely tokens through. Decode greedily when you measure, and sample with a temperature a little below 1 plus nucleus sampling when you want variety (briefing, section 7).

## Save the Adapter (used by Lab 8's stretch section)

`save_pretrained` writes only the adapter, $A$ and $B$ for every wrapped layer, plus the tokenizer with its chat template. The cell then checks two claims of the briefing on the full model: the adapter reloads onto a fresh base model with the same outputs, and `merge_and_unload` (eq. `lora-merge` in every wrapped layer) gives an ordinary model with the same outputs.

In [ ]:
ADAPTER_DIR = "lab07-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)  # includes chat_template.jinja
for path in sorted(Path(ADAPTER_DIR).iterdir()):
    print(f"{path.stat().st_size:>12,} bytes  {path.name}")

_ids, _attn = _probe[0].to(DEVICE), _probe[1].to(DEVICE)
reloaded = PeftModel.from_pretrained(load_base(), ADAPTER_DIR).eval()
with torch.no_grad():
    logits_trained = model(input_ids=_ids, attention_mask=_attn).logits
    logits_reloaded = reloaded(input_ids=_ids, attention_mask=_attn).logits
    merged = reloaded.merge_and_unload()  # a plain model: W_0 + (alpha / r) B A in each wrapped layer
    logits_merged = merged(input_ids=_ids, attention_mask=_attn).logits
reload_diff = (logits_reloaded - logits_trained).abs().max().item()
merge_diff = (logits_merged - logits_trained).abs().max().item()
print(f"\nmax |logits| difference: reloaded {reload_diff:.2e} · merged {merge_diff:.2e}{NOTE}")
print("merged model type:", type(merged).__name__, "· LoRA modules left:",
      sum("lora" in name for name, _ in merged.named_modules()))
assert reload_diff < 1e-5 and merge_diff < 1e-3
del reloaded, merged

In [ ]:
# Optional, Colab only: keep the adapter. A Colab runtime does not carry files into another notebook.
SAVE_TO_DRIVE = False  # copy to Google Drive (asks for permission)
ZIP_FOR_DOWNLOAD = False  # download a zip file

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    shutil.copytree(ADAPTER_DIR, "/content/drive/MyDrive/lab07-adapter", dirs_exist_ok=True)
    print("Copied to MyDrive/lab07-adapter")
if ZIP_FOR_DOWNLOAD:
    archive = shutil.make_archive("lab07-adapter", "zip", ADAPTER_DIR)
    if IN_COLAB:
        from google.colab import files

        files.download(archive)
    print("Wrote", archive)

## Results Card

Every number below was measured in this run. The cell also writes them to `lab07_results.json`.

In [ ]:
TIMES["total so far"] = time.time() - LAB_START
results = {
    "offline_test_stand_in": OFFLINE or None,  # if set, nothing here is a result
    "device": DEVICE,
    "fast": FAST,
    "model": MODEL_ID if not OFFLINE else MODEL_SOURCE,
    "revision": MODEL_REVISION if not OFFLINE else None,
    "subset": subset_stats,
    "run": RUN,
    "lora": {"r": lora_config.r, "lora_alpha": lora_config.lora_alpha,
             "target_modules": sorted(lora_config.target_modules), "trainable": trainable, "total": total},
    "heldout_examples": len(held_out),
    "heldout_response_tokens": n_resp_tokens,
    "before": {"response_loss": round(loss_before, 4), "response_ppl": round(math.exp(loss_before), 2),
               "endings": ends_before, "rouge1": round(rouge1_before, 4)},
    "after": {"response_loss": round(loss_after, 4), "response_ppl": round(math.exp(loss_after), 2),
              "endings": ends_after, "rouge1": round(rouge1_after, 4)},
    "seconds": {k: round(v, 1) for k, v in TIMES.items()},
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__},
}
Path("lab07_results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))
print("\nRun time by stage:")
for stage, seconds in TIMES.items():
    print(f"  {stage:<22} {seconds:>7.1f} s")
print("\nThis is the end of the core path." + NOTE)

## Stretch (optional) · How Large a Rank?

For those who finish early. Nothing later in the workshop depends on it.

Train four adapters with $r \in \{1, 4, 16, 64\}$ and `lora_alpha = 2 * r`, with the same seed, data and number of steps (`RUN["stretch_steps"]`, fewer than the main run), each starting from a fresh copy of the base model. Then plot held-out response loss against the number of trainable parameters.

**Predict.** The trainable count grows in proportion to $r$. Will the held-out loss fall in proportion too, or level off? Where?

**Task.** Run the two cells. The second trains one more adapter at $r = 8$ on every linear layer (`target_modules="all-linear"`) for as many steps as the main run (`RUN["steps"]`), and compares how its replies end with the $r = 8$ attention-only adapter of the main run.

In [ ]:
SWEEP_RANKS = [1, 4, 16, 64]
sweep = {}
started = time.time()
for rank in SWEEP_RANKS:
    base = load_base()
    expected = lora_param_count(base, rank, ["q_proj", "v_proj"])
    torch.manual_seed(SEED)
    sweep_model = get_peft_model(
        base, LoraConfig(r=rank, lora_alpha=2 * rank, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
    )
    count = sweep_model.get_nb_trainable_parameters()[0]
    assert count == expected, (rank, count, expected)
    if not OFFLINE or OFFLINE == "smollm2-shape":
        assert count == 460_800 * rank // 8, (rank, count)
    train_lora(sweep_model, train_set, RUN["stretch_steps"], log_every=0)
    sweep[rank] = (count, response_loss(sweep_model, held_out)[0])
    print(f"r = {rank:>2}: {count:>9,} trainable parameters · held-out response loss {sweep[rank][1]:.4f}{NOTE}")
    del sweep_model, base
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
TIMES["stretch"] = time.time() - started
print(f"({TIMES['stretch']:.1f} s for {len(SWEEP_RANKS)} runs of {RUN['stretch_steps']} steps)")

fig, ax = plt.subplots(figsize=(6, 3.6))
counts, losses = zip(*(sweep[rank] for rank in SWEEP_RANKS))
ax.plot(counts, losses, "o-", color=NAVY)
for rank, x, y in zip(SWEEP_RANKS, counts, losses):
    ax.annotate(f"r = {rank}", (x, y), textcoords="offset points", xytext=(5, 5))
ax.axhline(loss_before, color=ACCENT, linestyle="--", label="base model, before tuning")
ax.set_xscale("log")
ax.set_xlabel("trainable parameters (log scale)")
ax.set_ylabel("held-out response loss (nats)")
ax.set_title(f"{RUN['stretch_steps']} steps per rank" + (" [TEST-ONLY stand-in]" if OFFLINE else ""))
ax.legend(frameon=False)
plt.show()

**Explain.** The parameter counts are exact: 64 times as many at $r = 64$ as at $r = 1$. Read the loss curve against them. Where it flattens, extra rank buys little for this task, this many steps and this learning rate. Because $\alpha = 2r$ keeps the scale $\alpha / r$ fixed at 2, the ranks are compared at the same effective step size. With so few steps the comparison is noisy: try a second seed before you trust a small difference.

In [ ]:
# Every linear layer of every block (attention and feed-forward), r = 8, same steps and seed.
started = time.time()
base = load_base()
expected = lora_param_count(base, 8, ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
torch.manual_seed(SEED)
wide = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM"))
wide_count = wide.get_nb_trainable_parameters()[0]
assert wide_count == expected, (wide_count, expected)  # "all-linear" leaves out the output layer
train_lora(wide, train_set, RUN["steps"], log_every=0)
wide_loss = response_loss(wide, held_out)[0]
wide_ends = ending_rates(generate_batch(wide, gen_set)[1])
TIMES["stretch, all-linear"] = time.time() - started
print(f"{'r = 8':<22}{'trainable':>11}{'held-out loss':>15}{'ends <|im_end|>':>17}{'<|endoftext|>':>15}{'length':>8}")
for name, count, loss, ends in (("q_proj, v_proj (main)", trainable, loss_after, ends_after),
                                ("all-linear", wide_count, wide_loss, wide_ends)):
    print(f"{name:<22}{count:>11,}{loss:>15.4f}{ends['im_end']:>17.0%}{ends['endoftext']:>15.0%}{ends['length']:>8.0%}")
print(f"({TIMES['stretch, all-linear']:.1f} s){NOTE}")
del wide, base

**Explain.** Adapting every linear layer trains about 5.3 times as many parameters: 2,442,240 at $r = 8$ in this model, 1.8% of it, against 460,800. Compare the two rows: does the wider adapter fit the held-out replies better, and does it end more of them with `<|im_end|>`? If it does, the earlier attention-only adapter was limited in *where* it could change the model, not only in how much.

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [8 · LLMs Through APIs](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/08-llm-apis.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT